# 🕸️ BeautifulSoup4(BS4) 입문

---

**BeautifulSoup4** 는 HTML/XML 문서를 파이썬으로 읽고, 원하는 요소를 쉽게 찾을 수 있게 해주는 라이브러리입니다.  
`requests` 로 페이지를 가져오고, `BeautifulSoup` 로 파싱하는 조합이 가장 기본적인 스크래핑 흐름입니다.

## 📋 전체 학습 흐름

```
Part 1. 기본 준비
  1-1. 라이브러리 설치 및 import
  1-2. requests로 페이지 가져오기
  1-3. BeautifulSoup 객체 만들기

Part 2. 요소 탐색
  2-1. find() vs find_all() vs select()
  2-2. CSS 선택자 완전 정리
  2-3. 속성(attribute) 접근 방법
  2-4. 텍스트 추출 방법 비교
  2-5. 탐색 방향 (부모/자식/형제)

Part 3. 데이터 정제
  3-1. 텍스트 정제 (re 활용)
  3-2. 링크 처리 (urljoin)
  3-3. 안전한 추출 패턴

Part 4. 실전 예제
  4-1. 연습 사이트 크롤링 (목록 + 표)
  4-2. 네이버 뉴스 단일 기사 파싱
  4-3. 네이버 뉴스 다중 카테고리 수집
  4-4. 결과 저장 (CSV)

Part 5. 자주 하는 실수 & 팁
```

⚠️ 크롤링 전 반드시 해당 사이트의 **robots.txt** 와 **이용약관**을 확인하세요.


In [5]:
!pip install beautifulsoup4
!pip install requests


   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   -------------------------------- ------- 4/5 [requests]
   ---------------------------------------- 5/5 [requests]



In [6]:
# 필요한 라이브러리를 불러옵니다.
# requests      : 웹 페이지에 HTTP 요청을 보낼 때 사용
# BeautifulSoup : HTML을 파싱하고 원하는 요소를 찾을 때 사용
# pandas        : 수집 결과를 표 형태로 정리할 때 사용
# urljoin       : 상대경로 링크를 절대경로로 변환할 때 사용
# re            : 텍스트 정제에 정규표현식을 사용할 때
# time          : 요청 사이 대기 시간(크롤링 예절)을 줄 때

import re
import time
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

SITE_URL = 'https://crawl-da.netlify.app'
print('라이브러리 로드 완료!')


라이브러리 로드 완료!


---

# 🔧 Part 1. 기본 준비


## 1-1. requests로 페이지 가져오기

`requests.get()` 은 지정한 URL에 HTTP GET 요청을 보내고 응답을 받아옵니다.

| 속성/메서드                   | 설명                                          |
| :---------------------------- | :-------------------------------------------- |
| `response.status_code`        | 응답 상태 코드 (200=성공, 404=없음, 403=금지) |
| `response.text`               | HTML 문자열                                   |
| `response.content`            | 바이트 형식 (이미지 등)                       |
| `response.encoding`           | 인코딩 방식                                   |
| `response.raise_for_status()` | 오류 상태코드면 예외 발생 (안전장치)          |

> 💡 `timeout` 은 항상 설정하는 습관을 들이세요. 응답이 없는 서버를 무한정 기다리는 것을 방지합니다.


In [7]:
# 연습 사이트에 요청을 보냅니다.
response = requests.get("https://www.koreabaseball.com/", timeout=5)

# 상태 코드 확인
print('상태 코드:', response.status_code)    # 200 = 정상
print('인코딩:   ', response.encoding)
print('HTML 길이:', len(response.text), '자')
print()
print('HTML 앞부분 미리보기:')
print(response.text[:200])


상태 코드: 200
인코딩:    utf-8
HTML 길이: 113268 자

HTML 앞부분 미리보기:


<!DOCTYPE html>
<html lang="ko">
<head><title>
	메인 | KBO
</title><meta http-equiv="Content-Type" content="text/html; charset=utf-8" /><meta http-equiv="Content-Script-Type" content="text/javas


## 1-2. BeautifulSoup 객체 만들기

HTML 문자열을 `BeautifulSoup` 에 넘기면 파싱 가능한 객체가 됩니다.

| 파서          | 특징                                                     | 설치                   |
| :------------ | :------------------------------------------------------- | :--------------------- |
| `html.parser` | Python 내장, 별도 설치 불필요, 속도 보통                 | 불필요                 |
| `lxml`        | 빠르고 관대한 파싱, `:nth-child` 등 CSS 의사 클래스 지원 | `pip install lxml`     |
| `html5lib`    | 가장 관대하게 파싱, 느림                                 | `pip install html5lib` |

### ⚠️ 파서별 CSS 선택자 지원 범위

| CSS 선택자                               | html.parser | lxml |
| :--------------------------------------- | :---------: | :--: |
| `#id`, `.class`, `태그`                  |     ✅      |  ✅  |
| `[attr^=값]`, `[attr$=값]`, `[attr*=값]` |     ✅      |  ✅  |
| `:first-child`                           |     ✅      |  ✅  |
| `:last-child`                            |     ❌      |  ✅  |
| `:nth-child(n)`                          |     ❌      |  ✅  |

> 💡 **`html.parser` 에서 `:last-child`, `:nth-child()` 를 쓰면 오류가 발생합니다.**  
> 이 경우 `select()` 로 리스트를 받아 인덱싱으로 대체하세요.
>
> ```python
> rows = soup.select('tbody tr')
> rows[0]   # 첫 번째
> rows[-1]  # 마지막
> rows[1]   # 두 번째
> ```


In [8]:
html = response.text
soup = BeautifulSoup(html, 'html.parser')

# soup 객체 확인
print('타입:', type(soup))
print()

# prettify(): 들여쓰기가 적용된 HTML 출력 (디버깅에 유용)
print('prettify 앞부분:')
print(soup.prettify()[:300])


타입: <class 'bs4.BeautifulSoup'>

prettify 앞부분:
<!DOCTYPE html>
<html lang="ko">
 <head>
  <title>
   메인 | KBO
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="text/javascript" http-equiv="Content-Script-Type"/>
  <meta content="text/css" http-equiv="Content-Style-Type"/>
  <meta content="IE=edge,


---

# 🔍 Part 2. 요소 탐색


## 2-1. find() vs find_all() vs select()

요소를 찾는 방법은 크게 두 가지 계열이 있습니다.

| 메서드               | 반환          | 설명                          |
| :------------------- | :------------ | :---------------------------- |
| `find(태그)`         | Tag 또는 None | **첫 번째** 일치 요소 하나    |
| `find_all(태그)`     | 리스트        | 일치하는 **모든** 요소        |
| `select_one(선택자)` | Tag 또는 None | CSS 선택자로 **첫 번째** 요소 |
| `select(선택자)`     | 리스트        | CSS 선택자로 **모든** 요소    |

```
find()     ≈  select_one()  → 하나
find_all() ≈  select()      → 여럿
```

> 💡 `find/find_all` 은 태그 이름·속성으로 검색하고,  
> `select/select_one` 은 CSS 선택자 문법을 사용합니다.  
> 복잡한 구조에는 CSS 선택자가 더 간결합니다.


In [9]:
# 실습용 HTML
demo_html = '''
<div id="container">
  <h2 class="title">파이썬 기초</h2>
  <ul class="item_list">
    <li class="item">변수</li>
    <li class="item active">함수</li>
    <li class="item">클래스</li>
  </ul>
  <a href="/python">자세히 보기</a>
</div>
'''
dsoup = BeautifulSoup(demo_html, 'html.parser')

print('=== find() vs select_one() ===')
print(dsoup.find('h2'))               # <h2 class="title">파이썬 기초</h2>
print(dsoup.select_one('.title'))     # 동일 결과

print()

print('=== find_all() vs select() ===')
items_find   = dsoup.find_all('li', class_='item')
items_select = dsoup.select('.item_list .item')
print('find_all 결과 개수:', len(items_find))
print('select   결과 개수:', len(items_select))

print()

print('=== 텍스트 목록 ===')
for li in items_select:
    print(' -', li.get_text(strip=True))

print()

# find_all 에서 조건 추가
print('=== active 클래스 요소 ===')
active = dsoup.select('.item.active')
print([el.get_text(strip=True) for el in active])


=== find() vs select_one() ===
<h2 class="title">파이썬 기초</h2>
<h2 class="title">파이썬 기초</h2>

=== find_all() vs select() ===
find_all 결과 개수: 3
select   결과 개수: 3

=== 텍스트 목록 ===
 - 변수
 - 함수
 - 클래스

=== active 클래스 요소 ===
['함수']


### 🧩 확인 문제 2-1

아래 HTML 문자열을 BeautifulSoup 으로 파싱하고,  
페이지 `<title>` 텍스트를 출력하세요.

```
예상 결과: 제목: 테스트 페이지
```


In [ ]:
sample_html = '''
<html>
  <head><title>테스트 페이지</title></head>
  <body><h1>안녕하세요</h1></body>
</html>

# ✅ 정답
sample_soup = BeautifulSoup(sample_html, 'html.parser')
title = sample_soup.find('title')
print('제목:', title.get_text())

# 💡 해설:
# BeautifulSoup(html문자열, 파서) → soup 객체 생성
# soup.find('태그명') → 해당 태그를 찾아 Tag 객체 반환
# .get_text() → Tag 안의 텍스트만 추출


NameError: name 'sample_html' is not defined

## 2-2. CSS 선택자 완전 정리

`select()` / `select_one()` 에서 사용하는 CSS 선택자 문법입니다.

| 선택자        | 의미                 | 예시               | html.parser 지원 |
| :------------ | :------------------- | :----------------- | :--------------: |
| `태그`        | 태그 이름            | `div`, `p`, `a`    |        ✅        |
| `#id`         | id 속성              | `#main_title`      |        ✅        |
| `.class`      | class 속성           | `.article_item`    |        ✅        |
| `부모 자손`   | 부모 안의 모든 자손  | `#wrap .title`     |        ✅        |
| `부모 > 자식` | 바로 아래 자식만     | `ul > li`          |        ✅        |
| `A + B`       | A 바로 다음 형제 B   | `h2 + p`           |        ✅        |
| `A ~ B`       | A 이후 모든 형제 B   | `h2 ~ p`           |        ✅        |
| `[속성]`      | 속성 존재 여부       | `a[href]`          |        ✅        |
| `[속성=값]`   | 속성 값 일치         | `input[type=text]` |        ✅        |
| `[속성^=값]`  | 속성 값이 ~으로 시작 | `a[href^="https"]` |        ✅        |
| `[속성$=값]`  | 속성 값이 ~으로 끝   | `a[href$=".pdf"]`  |        ✅        |
| `[속성*=값]`  | 속성 값에 ~포함      | `a[href*="naver"]` |        ✅        |

**⚠️ 속성 선택자 값 따옴표 규칙**

속성값에 점(`.`), 슬래시(`/`) 등 특수문자가 포함되면 반드시 따옴표로 감싸야 합니다.

```python
# ❌ SelectorSyntaxError 발생
soup.select('a[href$=.pdf]')
soup.select('a[href^=https]')   # 단순 문자열은 동작하지만 권장하지 않음

# ✅ 값을 항상 따옴표로 감싸는 습관
soup.select('a[href$=".pdf"]')
soup.select('a[href^="https"]')
```

| `:first-child` | 첫 번째 자식 | `li:first-child` | ✅ |
| `:last-child` | 마지막 자식 | `li:last-child` | ❌ lxml만 |
| `:nth-child(n)` | n번째 자식 | `tr:nth-child(2)` | ❌ lxml만 |

### `:last-child`, `:nth-child()` 대체 방법 (html.parser)

```python
# ❌ html.parser 에서는 동작하지 않음
# soup.select_one('tbody tr:last-child')
# soup.select_one('tbody tr:nth-child(2)')

# ✅ select()로 리스트를 받아 인덱싱으로 대체
rows = soup.select('tbody tr')
rows[0]   # 첫 번째
rows[-1]  # 마지막
rows[1]   # 두 번째 (인덱스 = n-1)
```


In [12]:
selector_html = '''
<table id="price_table">
  <thead><tr><th>상품</th><th>가격</th><th>링크</th></tr></thead>
  <tbody>
    <tr><td class="name">사과</td><td class="price">3000원</td><td><a href="/fruit/apple">상세</a></td></tr>
    <tr><td class="name">바나나</td><td class="price">1500원</td><td><a href="/fruit/banana">상세</a></td></tr>
    <tr><td class="name">포도</td><td class="price">5000원</td><td><a href="/fruit/grape">상세</a></td></tr>
  </tbody>
</table>
<div class="promo">
  <a href="https://sale.example.com/fruit">과일 할인 이벤트</a>
  <a href="/about">소개 페이지</a>
  <a href="https://sale.example.com/veggie.pdf">채소 카탈로그(PDF)</a>
</div>
'''
tsoup = BeautifulSoup(selector_html, 'html.parser')

# html.parser 에서는 select()로 리스트를 받아 인덱싱으로 처리
rows = tsoup.select('#price_table tbody tr')

print('상품명 전체:', [el.get_text() for el in tsoup.select('#price_table .name')])
print('첫 번째 행:', rows[0].get_text(separator=' | ', strip=True))
print('마지막 행: ', rows[-1].get_text(separator=' | ', strip=True))   # rows[-1] = 마지막
print('두 번째 행:', rows[1].get_text(separator=' | ', strip=True))    # rows[1]  = 두 번째

print()

# ── 속성 선택자 (html.parser 에서 정상 동작) ─────────────────────
# ⚠️ 속성값에 점(.) 등 특수문자가 포함되면 반드시 따옴표로 감싸야 합니다.
#    a[href$=.pdf]   → ❌ SelectorSyntaxError
#    a[href$=".pdf"] → ✅ 정상 동작
print('href 있는 링크:', [a.get_text() for a in tsoup.select('a[href]')])
print('https로 시작:',   [a['href'] for a in tsoup.select('a[href^="https"]')])
print('.pdf로 끝남:',    [a['href'] for a in tsoup.select('a[href$=".pdf"]')])
print('sale 포함:',      [a['href'] for a in tsoup.select('a[href*="sale"]')])


상품명 전체: ['사과', '바나나', '포도']
첫 번째 행: 사과 | 3000원 | 상세
마지막 행:  포도 | 5000원 | 상세
두 번째 행: 바나나 | 1500원 | 상세

href 있는 링크: ['상세', '상세', '상세', '과일 할인 이벤트', '소개 페이지', '채소 카탈로그(PDF)']
https로 시작: ['https://sale.example.com/fruit', 'https://sale.example.com/veggie.pdf']
.pdf로 끝남: ['https://sale.example.com/veggie.pdf']
sale 포함: ['https://sale.example.com/fruit', 'https://sale.example.com/veggie.pdf']


### 🧩 확인 문제 2-2

위 `selector_html` 에서 `tbody` 안의 두 번째 행에 있는 **가격**만 추출하세요.

```
예상 결과: 1500원
```


In [13]:
# ✅ 정답
# html.parser 에서는 :nth-child() 대신 리스트 인덱싱 사용
rows = tsoup.select('#price_table tbody tr')
price = rows[1].select_one('.price')   # rows[1] = 두 번째 행
print(price.get_text(strip=True))

# 💡 해설:
# rows[1] → 두 번째 tr (0-based 인덱스)
# .select_one('.price') → 그 안의 class='price' 요소
# ":nth-child(2)" 를 쓰면 html.parser 에서 오류 발생


1500원


## 2-3. 속성(attribute) 접근 방법

태그의 `href`, `src`, `class`, `id` 등 속성에 접근하는 방법은 세 가지입니다.

| 방법                          | 설명                   | 없을 때       |
| :---------------------------- | :--------------------- | :------------ |
| `tag['속성명']`               | 딕셔너리 방식          | KeyError 발생 |
| `tag.get('속성명')`           | 안전한 방식            | None 반환     |
| `tag.get('속성명', '기본값')` | 기본값 지정            | 기본값 반환   |
| `tag.attrs`                   | 모든 속성을 딕셔너리로 | —             |

> ✅ 사용시에는 **`tag.get('속성명')`** 을 쓰는 것이 가장 안전합니다.


In [15]:
attr_html = '''
<div>
  <a href="https://www.koreabaseball.com" class="link external" target="_blank">외부 링크</a>
  <img src="/images/photo.jpg" alt="사진" width="300">
  <input type="text" name="search" placeholder="검색어를 입력하세요">
  <a class="no-href">href 없는 링크</a>
</div>
'''
asoup = BeautifulSoup(attr_html, 'html.parser')

link = asoup.find('a', class_='external')

print('=== 속성 접근 방법 비교 ===')
print('딕셔너리 방식:  ', link['href'])
print('get() 방식:     ', link.get('href'))
print('없는 속성 get():', link.get('data-id'))          # None (오류 없음)
print('기본값 지정:    ', link.get('data-id', 'N/A'))    # N/A
print('모든 속성:      ', link.attrs)

print()

# class 는 리스트로 반환됨 (여러 클래스 가능)
print('class 속성 타입:', type(link.get('class')))  # list
print('class 목록:     ', link.get('class'))

print()

# 이미지 속성
img = asoup.find('img')
print('=== 이미지 속성 ===')
print('src:  ', img.get('src'))
print('alt:  ', img.get('alt'))
print('width:', img.get('width'))

print()

# href 없는 요소에 안전하게 접근
no_href_link = asoup.find('a', class_='no-href')
print('href 없는 링크 (get):    ', no_href_link.get('href'))   # None
# print(no_href_link['href'])  # ← KeyError 발생!


=== 속성 접근 방법 비교 ===
딕셔너리 방식:   https://www.koreabaseball.com
get() 방식:      https://www.koreabaseball.com
없는 속성 get(): None
기본값 지정:     N/A
모든 속성:       {'href': 'https://www.koreabaseball.com', 'class': ['link', 'external'], 'target': '_blank'}

class 속성 타입: <class 'bs4.element.AttributeValueList'>
class 목록:      ['link', 'external']

=== 이미지 속성 ===
src:   /images/photo.jpg
alt:   사진
width: 300

href 없는 링크 (get):     None


### 🧩 확인 문제 2-3

아래 HTML에서 모든 `<img>` 태그의 `src` 와 `alt` 를 딕셔너리 리스트로 만드세요.

```
예상 결과: [{'src': '/img/a.jpg', 'alt': '이미지A'}, {'src': '/img/b.png', 'alt': '이미지B'}, {'src': '/img/c.gif', 'alt': 'no-alt'}]
단, alt 속성이 없는 경우 'no-alt' 로 표시하세요.
```


In [16]:
img_html = '''
<div>
  <img src="/img/a.jpg" alt="이미지A">
  <img src="/img/b.png" alt="이미지B">
  <img src="/img/c.gif">
</div>
'''
isoup = BeautifulSoup(img_html, 'html.parser')

# ✅ 정답
result = [
    {'src': img.get('src'), 'alt': img.get('alt', 'no-alt')}
    for img in isoup.find_all('img')
]
print(result)

# 💡 해설:
# find_all('img') → 모든 img 태그 리스트
# img.get('alt', 'no-alt') → alt 없으면 'no-alt' 반환
# 리스트 컴프리헨션으로 한 줄 정리


[{'src': '/img/a.jpg', 'alt': '이미지A'}, {'src': '/img/b.png', 'alt': '이미지B'}, {'src': '/img/c.gif', 'alt': 'no-alt'}]


## 2-4. 텍스트 추출 방법 비교

텍스트를 꺼내는 방법은 여러 가지이며, 상황에 따라 적절한 것을 선택해야 합니다.

| 방법                       | 설명                                   |
| :------------------------- | :------------------------------------- |
| `.get_text()`              | 태그 내부 모든 텍스트를 하나로 합침    |
| `.get_text(strip=True)`    | 앞뒤 공백 제거                         |
| `.get_text(separator=' ')` | 구분자로 텍스트를 이어 붙임            |
| `.string`                  | 태그 안에 직접 텍스트가 하나만 있을 때 |
| `.strings`                 | 태그 안 모든 텍스트 조각 이터레이터    |
| `.stripped_strings`        | 공백 제거된 텍스트 조각 이터레이터     |


In [17]:
text_html = '''
<div class="article">
  <h2>  파이썬 크롤링  </h2>
  <p>웹에서 데이터를 <strong>자동으로</strong> 수집하는 기술입니다.<br>초보자도 쉽게 배울 수 있습니다.</p>
  <span>  </span>
</div>
'''
txsoup = BeautifulSoup(text_html, 'html.parser')
div = txsoup.find('div')
p   = txsoup.find('p')

print('=== get_text() 비교 ===')
print('기본:          ', repr(div.get_text()))
print('strip=True:    ', repr(div.get_text(strip=True)))
print('separator=" " :', repr(div.get_text(separator=' ', strip=True)))

print()

print('=== .string ===')
h2 = txsoup.find('h2')
print('.string h2:', repr(h2.string))   # 텍스트가 하나면 반환
print('.string p: ', repr(p.string))    # 자식 태그 있으면 None

print()

print('=== stripped_strings ===')
# <br> 태그 있어도 각 줄을 개별 텍스트로 추출
for piece in p.stripped_strings:
    print(' -', repr(piece))

print()

print('=== 실제 패턴: stripped_strings 를 join으로 합치기 ===')
clean_text_ex = ' '.join(p.stripped_strings)
print(clean_text_ex)


=== get_text() 비교 ===
기본:           '\n  파이썬 크롤링  \n웹에서 데이터를 자동으로 수집하는 기술입니다.초보자도 쉽게 배울 수 있습니다.\n \n'
strip=True:     '파이썬 크롤링웹에서 데이터를자동으로수집하는 기술입니다.초보자도 쉽게 배울 수 있습니다.'
separator=" " : '파이썬 크롤링 웹에서 데이터를 자동으로 수집하는 기술입니다. 초보자도 쉽게 배울 수 있습니다.'

=== .string ===
.string h2: '  파이썬 크롤링  '
.string p:  None

=== stripped_strings ===
 - '웹에서 데이터를'
 - '자동으로'
 - '수집하는 기술입니다.'
 - '초보자도 쉽게 배울 수 있습니다.'

=== 실제 패턴: stripped_strings 를 join으로 합치기 ===
웹에서 데이터를 자동으로 수집하는 기술입니다. 초보자도 쉽게 배울 수 있습니다.


## 2-5. 탐색 방향 — 부모 / 자식 / 형제

특정 요소를 기준으로 주변을 탐색할 수 있습니다.

| 방향       | 속성/메서드                    | 설명                       |
| :--------- | :----------------------------- | :------------------------- |
| 자식       | `.children`                    | 바로 아래 자식들           |
| 자식       | `.descendants`                 | 모든 자손                  |
| 부모       | `.parent`                      | 바로 위 부모               |
| 부모       | `.parents`                     | 모든 조상                  |
| 형제(이전) | `.find_previous_sibling(태그)` | 앞 형제 중 첫 번째 매칭    |
| 형제(다음) | `.find_next_sibling(태그)`     | 뒤 형제 중 첫 번째 매칭    |
| 검색       | `.find_parent(태그)`           | 위로 올라가며 첫 번째 매칭 |


In [18]:
nav_html = '''
<section id="s1">
  <h3>섹션 제목</h3>
  <ul>
    <li id="item1">첫 번째</li>
    <li id="item2" class="active">두 번째 (현재)</li>
    <li id="item3">세 번째</li>
  </ul>
</section>
'''
nsoup = BeautifulSoup(nav_html, 'html.parser')

current = nsoup.find('li', class_='active')
print('현재 요소:   ', current.get_text(strip=True))

print()

# 형제 탐색
prev_sib = current.find_previous_sibling('li')
next_sib = current.find_next_sibling('li')
print('이전 형제:', prev_sib.get_text() if prev_sib else '없음')
print('다음 형제:', next_sib.get_text() if next_sib else '없음')

print()

# 부모 탐색
print('부모 태그:     ', current.parent.name)
print('find_parent(): ', current.find_parent('section').get('id'))

print()

# 자식 탐색
ul = nsoup.find('ul')
children = [c for c in ul.children if c.name]  # 텍스트 노드 제외
print('ul 자식 목록:', [c.get_text(strip=True) for c in children])


현재 요소:    두 번째 (현재)

이전 형제: 첫 번째
다음 형제: 세 번째

부모 태그:      ul
find_parent():  s1

ul 자식 목록: ['첫 번째', '두 번째 (현재)', '세 번째']


### 🧩 확인 문제 2-5

위 `nav_html` 에서 `id='item2'` 요소의 **다음 형제 li** 텍스트를 출력하세요.

```
예상 결과: 세 번째
```


In [13]:
# ✅ 정답
item2 = nsoup.find('li', id='item2')
next_item = item2.find_next_sibling('li')
print(next_item.get_text(strip=True))

# 💡 해설:
# find('li', id='item2') → id 속성으로 특정 요소 선택
# .find_next_sibling('li') → 같은 레벨의 다음 li 요소


세 번째


---

# 🧹 Part 3. 데이터 정제


## 3-1. 텍스트 정제 (re 활용)

크롤링한 텍스트에는 불필요한 공백, 특수문자, 광고 문구 등이 섞여 있을 수 있습니다.  
`re.sub()` 을 활용해 깔끔하게 정제하는 패턴을 익혀 봅니다.


In [19]:
def clean_text(text: str) -> str:
    if not text:
        return ''
    text = re.sub(r'\s+', ' ', text)           # 탭·줄바꿈·연속 공백 → 단일 공백
    text = re.sub(r'\[.*?\]', '', text)         # [광고], [사진] 등 대괄호 블록 제거
    text = re.sub(r'\(.*?\)\s*[\w\s]+기자\s*=\s*', '', text)
    # └ \(.*?\)      : (서울=뉴스1) 출처 괄호
    # └ \s*          : 괄호 뒤 공백
    # └ [\w\s]+      : 홍길동 등 기자 이름 (한글·영문·공백 허용)
    # └ 기자\s*=\s*  : '기자 =' 구분자까지 포함해 제거
    text = re.sub(r'<[^>]+>', '', text)         # <br>, <p> 등 잔여 HTML 태그 제거
    return text.strip()


def clean_price(text: str) -> int:
    """가격 문자열에서 숫자만 추출해 정수로 반환"""
    number = re.sub(r'[^0-9]', '', text)        # 숫자 외 모든 문자(쉼표·원·공백 등) 제거
    return int(number) if number else 0         # 추출 실패 시 0 반환


def clean_date(text: str) -> str:
    """날짜 문자열에서 YYYY.MM.DD 또는 YYYY-MM-DD 형식 추출"""
    m = re.search(r'\d{4}[.\-/]\d{1,2}[.\-/]\d{1,2}', text)
    # └ \d{4}        : 연도 4자리
    # └ [.\-/]       : 구분자 '.', '-', '/' 모두 허용
    # └ \d{1,2}      : 월·일은 1~2자리 허용 (예: 03 또는 3)
    return m.group() if m else ''               # 매칭 실패 시 빈 문자열 반환


# 확인
print('텍스트 정제:')
samples = [
    '[파이낸셜뉴스] 식약처가 희귀질환자 치료에 필요한   의료기기를 지정했다.',
    '(서울=뉴스1) 홍길동 기자 = 인공지능 기반 진단 전문기업이 해외 진출을 추진한다.',
]
for s in samples:
    print(f'  Before: {s}')
    print(f'  After : {clean_text(s)}')
    print()

print('가격 정제:', clean_price('18,000원 (VAT 포함)'))  # → 18000
print('날짜 정제:', clean_date('등록일: 2024-03-15 오전 10:30'))  # → 2024-03-15

텍스트 정제:
  Before: [파이낸셜뉴스] 식약처가 희귀질환자 치료에 필요한   의료기기를 지정했다.
  After : 식약처가 희귀질환자 치료에 필요한 의료기기를 지정했다.

  Before: (서울=뉴스1) 홍길동 기자 = 인공지능 기반 진단 전문기업이 해외 진출을 추진한다.
  After : 인공지능 기반 진단 전문기업이 해외 진출을 추진한다.

가격 정제: 18000
날짜 정제: 2024-03-15


## 3-2. 링크 처리 — `urljoin`

크롤링한 `href` 값은 절대경로(`https://...`)일 수도 있고 상대경로(`/article/1`)일 수도 있습니다.  
`urljoin(base_url, href)` 를 사용하면 **어떤 형태든 안전하게 절대경로로 변환**할 수 있습니다.

```
urljoin('https://example.com/news/', '/article/1')    → 'https://example.com/article/1'
urljoin('https://example.com/news/', 'article/1')     → 'https://example.com/news/article/1'
urljoin('https://example.com/news/', 'https://other.com') → 'https://other.com'
```


In [23]:
base = 'https://example.com/news/'
hrefs = [
    '/article/123',
    'article/456',
    '../images/photo.jpg',
    'https://other.com/page',
    '#section1',
    '',
]

print('urljoin 변환 결과:')
for href in hrefs:
    full = urljoin(base, href) if href else None
    print(f'  {href!r:35} -> {full}')

print()

# 같은 도메인 링크만 필터링하는 패턴
def is_same_domain(base_url, href):
    full = urljoin(base_url, href)
    return urlparse(full).netloc == urlparse(base_url).netloc

print('같은 도메인 필터링:')
for href in hrefs:
    if href:
        same = is_same_domain(base, href)
        print(f'  {href!r:35} -> {"✅ 같음" if same else "❌ 다른 도메인"}')


urljoin 변환 결과:
  '/article/123'                      -> https://example.com/article/123
  'article/456'                       -> https://example.com/news/article/456
  '../images/photo.jpg'               -> https://example.com/images/photo.jpg
  'https://other.com/page'            -> https://other.com/page
  '#section1'                         -> https://example.com/news/#section1
  ''                                  -> None

같은 도메인 필터링:
  '/article/123'                      -> ✅ 같음
  'article/456'                       -> ✅ 같음
  '../images/photo.jpg'               -> ✅ 같음
  'https://other.com/page'            -> ❌ 다른 도메인
  '#section1'                         -> ✅ 같음


## 3-3. 안전한 추출 패턴

크롤링 중 요소가 없거나 구조가 다를 때 프로그램이 멈추지 않도록 처리하는 방법입니다.

```python
# ❌ 위험한 방식 — 요소가 None이면 AttributeError!
title = soup.select_one('.title').get_text()

# ✅ 안전한 방식 — 조건 체크
el = soup.select_one('.title')
title = el.get_text(strip=True) if el else None
```


In [24]:
# 안전한 추출 헬퍼 함수 모음

def safe_text(tag, selector, default=None):
    """선택자로 요소를 찾아 텍스트 반환. 없으면 default."""
    el = tag.select_one(selector)
    return el.get_text(strip=True) if el else default

def safe_attr(tag, selector, attr, default=None):
    """선택자로 요소를 찾아 속성값 반환. 없으면 default."""
    el = tag.select_one(selector)
    return el.get(attr, default) if el else default

def safe_texts(tag, selector):
    """선택자로 여러 요소를 찾아 텍스트 리스트 반환."""
    return [el.get_text(strip=True) for el in tag.select(selector)]

# 테스트
test_html = '''
<article>
  <h2 class="title">뉴스 제목</h2>
  <a href="/detail/1" class="link">읽기</a>
</article>
'''
tsoup2 = BeautifulSoup(test_html, 'html.parser')

print('safe_text 결과:')
print('  제목:       ', safe_text(tsoup2, '.title'))
print('  없는 요소:  ', safe_text(tsoup2, '.summary', default='(요약 없음)'))

print('safe_attr 결과:')
print('  링크 href:  ', safe_attr(tsoup2, '.link', 'href'))
print('  없는 속성:  ', safe_attr(tsoup2, '.title', 'href', default='#'))

print()
print('에러 처리 패턴:')
urls_to_check = ['https://crawl-da.netlify.app', 'https://nonexistent999abc.xyz']
for url in urls_to_check:
    try:
        resp = requests.get(url, timeout=3)
        resp.raise_for_status()
        print(f'  ✅ {url} — 상태 {resp.status_code}')
    except requests.exceptions.ConnectionError:
        print(f'  ❌ {url} — 연결 실패')
    except requests.exceptions.Timeout:
        print(f'  ❌ {url} — 시간 초과')
    except requests.exceptions.HTTPError as e:
        print(f'  ❌ {url} — HTTP 오류: {e}')


safe_text 결과:
  제목:        뉴스 제목
  없는 요소:   (요약 없음)
safe_attr 결과:
  링크 href:   /detail/1
  없는 속성:   #

에러 처리 패턴:
  ✅ https://crawl-da.netlify.app — 상태 200
  ❌ https://nonexistent999abc.xyz — 연결 실패


### 🧩 확인 문제 3

아래 HTML에서 `safe_text`, `safe_attr` 함수를 활용해  
제목, 날짜, 링크 href 를 추출하세요.

```
예상 결과:
  제목: 크롤링 실습 기사
  날짜: None
  링크: /article/999
```


In [25]:
quiz_html = '''
<div class="card">
  <h3 class="card_title">크롤링 실습 기사</h3>
  <a href="/article/999" class="card_link">읽기</a>
</div>
'''
qsoup = BeautifulSoup(quiz_html, 'html.parser')

# ✅ 정답
title = safe_text(qsoup, '.card_title')
date  = safe_text(qsoup, '.card_date')          # 없는 요소 → None
link  = safe_attr(qsoup, '.card_link', 'href')

print('제목:', title)
print('날짜:', date)
print('링크:', link)

# 💡 해설:
# safe_text, safe_attr 함수를 쓰면 요소가 없어도 None 반환
# AttributeError 없이 안전하게 처리 가능


제목: 크롤링 실습 기사
날짜: None
링크: /article/999


---

# 🏋️ Part 4. 실전 예제


## 4-1. 연습 사이트 크롤링

[연습 사이트](https://crawl-da.netlify.app)를 대상으로  
목록 카드와 표 데이터를 수집하고 DataFrame 으로 정리합니다.


In [27]:
response = requests.get('https://news.naver.com/section/103', timeout=5)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')

main_title = safe_text(soup, '#main_title')
print('메인 제목:', main_title)


메인 제목: None


In [19]:
# ── 기사 카드 목록 → DataFrame ───────────────────────────────────
article_rows = []

for article in soup.select('#news_section .article_item'):
    title    = safe_text(article, '.article_title')
    href     = safe_attr(article, 'a.link_thumb', 'href')
    full_url = urljoin(SITE_URL, href) if href else None
    summary_el = article.select_one('.summary_text')
    summary  = ' '.join(summary_el.stripped_strings) if summary_el else None

    article_rows.append({'제목': title, '주소': full_url, '요약': summary})

article_df = pd.DataFrame(article_rows)
print('기사 카드 수집 결과:')
article_df


기사 카드 수집 결과:


,제목,주소,요약
0,파이썬 웹 크롤링 핵심 가이드,https://example.com/article/101,정적 페이지부터 동적 페이지까지 데이터 수집의 모든 것. 초보자도 쉽게 따라 할 수...
1,Pandas로 시작하는 데이터 전처리,https://example.com/article/102,수집한 데이터를 예쁘게 다듬어 봅시다. 결측치 처리부터 DataFrame 병합까지 ...


In [20]:
# ── 표(Table) 데이터 → DataFrame ─────────────────────────────────
# HTML 표 구조: table > thead/tbody > tr > th/td
# html.parser 에서는 :nth-child 대신 리스트 인덱싱 사용

book_rows = []
rows = soup.select('#book_table tbody tr')

for row in rows:
    rank       = safe_text(row, '.rank_num')
    name       = safe_text(row, '.book_name')
    author     = safe_text(row, '.author_name')
    price_text = safe_text(row, '.price_info', default='0')

    book_rows.append({
        '순위':  int(rank) if rank and rank.isdigit() else None,
        '도서명': name,
        '저자':  author,
        '가격':  clean_price(price_text),
    })

book_df = pd.DataFrame(book_rows)
print('도서 표 수집 결과:')
book_df


도서 표 수집 결과:


,순위,도서명,저자,가격
0,1,Do it! 점프 투 파이썬,박응용,18000
1,2,혼자 공부하는 머신러닝+딥러닝,박해선,26000
2,3,파이썬 라이브러리를 활용한 데이터 분석,웨스 맥키니,35000


### 🧩 확인 문제 4-1

위에서 수집한 `book_df` 에서 **가격이 20,000원 이상인 도서**만 필터링하여 출력하세요.


In [21]:
# ✅ 정답
expensive = book_df[book_df['가격'] >= 20000]
print(expensive.to_string(index=False))

# 💡 해설:
# 가격 컬럼이 int 타입이라 비교 연산자 바로 사용 가능
# clean_price() 에서 숫자만 추출해 int 로 변환했기 때문


 순위                   도서명     저자    가격
  2      혼자 공부하는 머신러닝+딥러닝    박해선 26000
  3 파이썬 라이브러리를 활용한 데이터 분석 웨스 맥키니 35000


## 4-2. 네이버 뉴스 단일 기사 파싱

실제 기사 페이지에서 **제목 / 요약 / 본문**을 나눠서 추출합니다.


In [28]:
news_url = 'https://n.news.naver.com/mnews/article/469/0000918291'

response = requests.get(news_url, timeout=10)
response.raise_for_status()
news_soup = BeautifulSoup(response.text, 'html.parser')

# 제목
title = safe_text(news_soup, '#title_area')

# 요약 (없는 기사도 있음)
summary_el = news_soup.select_one('.media_end_summary')
summary_list = list(summary_el.stripped_strings) if summary_el else []

# 본문
body_el = news_soup.select_one('#dic_area')
body_lines = list(body_el.stripped_strings) if body_el else []

# 본문 앞부분과 요약이 겹치는 경우 제거
if summary_list and body_lines[:len(summary_list)] == summary_list:
    body_lines = body_lines[len(summary_list):]

body_text = clean_text(' '.join(body_lines))

print('제목:', title)
print()
print('요약:')
for line in summary_list:
    print(' -', line)
print()
print('본문 앞 200자:')
print(body_text[:200])
print()
print('본문 길이:', len(body_text), '자')


제목: 점심엔 봄나물 비빔밥, 저녁엔 종어·해삼… 봄 메뉴로 돌아온 '한국의집'

요약:
 - 국가유산진흥원, 45년 만에 리모델링 마치고 11일 재개관
 - 조선 조리서 참고 전국 각지 식재료·봄나물 활용 식단 마련

본문 앞 200자:
'한국의집'에서 제공하는 대표적인 궁중 전채요리인 구절판. 9개로 나뉜 찬합의 가운데에 놓인 밀전병에 주변에 놓인 고기·채소를 싸 먹는다. 국가유산진흥원 제공 국가무형유산 '궁중음식'을 응용해 파인다이닝을 선보여 온 '한국의집'이 재개관과 함께 봄 메뉴를 새롭게 선보인다. 9일 한국의집을 운영하는 국가유산청 산하 국가유산진흥원에 따르면 한국의집은 리모델링을

본문 길이: 1466 자


## 4-3. 뉴스 파싱 함수화 & 다중 카테고리 수집

### 수집 파이프라인

| 단계        | 설명                                         |
| :---------- | :------------------------------------------- |
| ① URL 생성  | 카테고리 코드로 섹션 URL 동적 생성           |
| ② 링크 추출 | 목록 페이지에서 기사 URL 수집                |
| ③ 본문 파싱 | 각 기사 URL을 요청해 제목/요약/본문 추출     |
| ④ 통합      | 카테고리별 결과를 하나의 DataFrame 으로 합산 |


In [23]:
def parse_news_article(soup: BeautifulSoup, category: str = None) -> dict:
    """네이버 뉴스 기사 1개를 파싱하여 딕셔너리로 반환합니다."""
    title = safe_text(soup, '#title_area')

    summary_el = soup.select_one('.media_end_summary')
    summary_list = list(summary_el.stripped_strings) if summary_el else []

    body_el = soup.select_one('#dic_area')
    body_lines = list(body_el.stripped_strings) if body_el else []

    if summary_list and body_lines[:len(summary_list)] == summary_list:
        body_lines = body_lines[len(summary_list):]

    body_text = clean_text(' '.join(body_lines))

    result = {
        '제목':    title,
        '요약':    ' | '.join(summary_list) if summary_list else None,
        '본문':    body_text,
        '본문 길이': len(body_text),
    }
    if category:
        result['카테고리'] = category
    return result

print('함수 정의 완료')


함수 정의 완료


In [24]:
category_map = {
    '건강정보': '241',
    '음식/맛집': '238',
    '공연/전시': '242',
}

ARTICLE_SELECTOR  = '.section_latest_article .sa_item a.sa_thumb_link'
MAX_PER_CATEGORY  = 3
DELAY_SEC         = 0.5   # 요청 사이 대기 (크롤링 예절)

all_rows = []

for category_name, category_code in category_map.items():
    section_url = f'https://news.naver.com/breakingnews/section/103/{category_code}'

    try:
        resp = requests.get(section_url, timeout=10)
        resp.raise_for_status()
        sec_soup = BeautifulSoup(resp.text, 'html.parser')
        links = [el.get('href') for el in sec_soup.select(ARTICLE_SELECTOR)]
        links = links[:MAX_PER_CATEGORY]
    except Exception as e:
        print(f'  ❌ [{category_name}] 목록 수집 실패: {e}')
        continue

    for news_url in links:
        time.sleep(DELAY_SEC)
        try:
            resp = requests.get(news_url, timeout=10)
            resp.raise_for_status()
            art_soup = BeautifulSoup(resp.text, 'html.parser')
            parsed = parse_news_article(art_soup, category=category_name)
            parsed['링크'] = news_url
            all_rows.append(parsed)
        except Exception as e:
            all_rows.append({
                '카테고리': category_name, '제목': None,
                '요약': None, '본문': None, '본문 길이': 0,
                '링크': news_url, '오류': str(e),
            })

news_df = pd.DataFrame(all_rows)
print(f'총 수집: {len(news_df)}건')
news_df.head()


총 수집: 9건


,제목,요약,본문,본문 길이,카테고리,링크
0,당뇨병 걸리면 살 빠진다는데… 왜 나는 '비만 당뇨'지?,NaN,체중이 줄지 않는다고 안심할 것이 아니라 비만한 상태 자체가 당뇨병의 초기 신호일 ...,1586,건강정보,https://n.news.naver.com/mnews/article/346/000...
1,"“2년간 못 걸었다” 하희라, 아픔 극복하고 아침마다 ‘이것’ 실천…왜?",[셀럽헬스] 배우 하희라 아침 루틴,하희라는 20대 시절 촬영 중 오토바이에서 떨어져 등을 다쳤다. 이후 등부터 허리 ...,1391,건강정보,https://n.news.naver.com/mnews/article/296/000...
2,"""집 안 공기 점검해야""… 라돈, 난소암 위험 높인다",NaN,눈에 보이지 않는 방사성 가스 '라돈'이 여성의 난소암 위험을 높일 수 있다는 연구...,1313,건강정보,https://n.news.naver.com/mnews/article/346/000...
3,만세보령 삼광미골드' 대한민국 대표브랜드 쌀 부문 대상,7년 연속 수상,김동일 시장충남 보령시는 '만세보령 삼광미골드'가 2026년 대한민국 대표브랜드 쌀...,280,음식/맛집,https://n.news.naver.com/mnews/article/421/000...
4,"“60대부터 바꿔도 늦지 않다”…식물성 식단, 치매 위험 낮춘다",NaN,"최근 학술지 Neurology에 발표된 연구에 따르면 통곡물, 채소, 과일 등 식물...",1872,음식/맛집,https://n.news.naver.com/mnews/article/145/000...


## 4-4. 결과 저장

`utf-8-sig` 인코딩을 사용하면 Windows Excel에서 한글이 깨지지 않습니다.


In [25]:
output_path = 'naver_news_crawl.csv'
news_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {output_path}')
print(f'행 수: {len(news_df)}, 열 수: {len(news_df.columns)}')
print('컬럼:', list(news_df.columns))


저장 완료: naver_news_crawl.csv
행 수: 9, 열 수: 6
컬럼: ['제목', '요약', '본문', '본문 길이', '카테고리', '링크']


### 🧩 확인 문제 4-4

수집한 `news_df` 에서 다음을 출력하세요.

1. 카테고리별 기사 수
2. 본문 길이가 가장 긴 기사의 제목과 길이


In [26]:
# ✅ 정답

# 1. 카테고리별 기사 수
if '카테고리' in news_df.columns:
    print('카테고리별 기사 수:')
    print(news_df['카테고리'].value_counts().to_string())
    print()

# 2. 본문 길이 최대 기사
if '본문 길이' in news_df.columns and news_df['본문 길이'].notna().any():
    max_row = news_df.loc[news_df['본문 길이'].idxmax()]
    print('본문 가장 긴 기사:')
    print('  제목:', max_row['제목'])
    print('  길이:', max_row['본문 길이'], '자')

# 💡 해설:
# value_counts() → 카테고리별 등장 횟수
# idxmax()       → 최댓값 행의 인덱스 반환
# .loc[인덱스]   → 해당 행 전체 데이터 접근


카테고리별 기사 수:
카테고리
건강정보     3
음식/맛집    3
공연/전시    3

본문 가장 긴 기사:
  제목: “유통기한 지났다면”…절대 먹으면 안 되는 식품 10가지
  길이: 2136 자


---

# ⚠️ Part 5. 자주 하는 실수 & 팁


## 5-1. 자주 하는 실수 모음


In [27]:
print('=' * 60)
print('실수 1. None 체크 없이 메서드 호출')
print('=' * 60)
bad_html = '<div><p>텍스트</p></div>'
bsoup = BeautifulSoup(bad_html, 'html.parser')

# ❌ 없는 요소에 바로 .get_text() 호출 → AttributeError
# title = bsoup.select_one('.title').get_text()

# ✅ 올바른 방법
el = bsoup.select_one('.title')
title = el.get_text() if el else None
print('안전한 추출:', title)

print()

print('=' * 60)
print('실수 2. html.parser 에서 :nth-child, :last-child 사용')
print('=' * 60)
nth_html = '<ul><li>A</li><li>B</li><li>C</li></ul>'
nth_soup = BeautifulSoup(nth_html, 'html.parser')

# ❌ html.parser 에서는 오류 발생
# nth_soup.select_one('li:last-child')
# nth_soup.select_one('li:nth-child(2)')

# ✅ 리스트 인덱싱으로 대체
lis = nth_soup.select('li')
print('첫 번째:', lis[0].get_text())
print('두 번째:', lis[1].get_text())
print('마지막:', lis[-1].get_text())

print()

print('=' * 60)
print('실수 3. class_ 대신 class 사용')
print('=' * 60)
cls_html = '<p class="intro">안녕하세요</p>'
csoup = BeautifulSoup(cls_html, 'html.parser')
# ❌ find('p', class='intro')  → SyntaxError (class 는 Python 예약어)
# ✅ 올바른 방법
el = csoup.find('p', class_='intro')   # find() 에서는 class_
el2 = csoup.select_one('.intro')       # select 에서는 .class명
print('find(class_=):', el.get_text())
print('select(.):    ', el2.get_text())

print()

print('=' * 60)
print('실수 4. href 가 None 인데 urljoin 사용')
print('=' * 60)
href = None
# ❌ urljoin(base, None) → TypeError
# ✅ 올바른 방법
full_url = urljoin(SITE_URL, href) if href else None
print('안전한 urljoin:', full_url)

print()

print('=' * 60)
print('실수 5. find_all 결과 바로 인덱싱')
print('=' * 60)
items = bsoup.find_all('.nonexistent')
# ❌ items[0] → IndexError
# ✅ 올바른 방법
if items:
    print(items[0])
else:
    print('결과 없음 — 안전하게 처리')

print()

print('=' * 60)
print('실수 6. 인코딩 문제')
print('=' * 60)
print('한국어 사이트 중 일부는 euc-kr 인코딩 사용')
print('response.encoding 확인 후 필요하면 직접 지정:')
print('  response.encoding = "euc-kr"')
print('  또는')
print('  text = response.content.decode("euc-kr")')


실수 1. None 체크 없이 메서드 호출
안전한 추출: None

실수 2. html.parser 에서 :nth-child, :last-child 사용
첫 번째: A
두 번째: B
마지막: C

실수 3. class_ 대신 class 사용
find(class_=): 안녕하세요
select(.):     안녕하세요

실수 4. href 가 None 인데 urljoin 사용
안전한 urljoin: None

실수 5. find_all 결과 바로 인덱싱
결과 없음 — 안전하게 처리

실수 6. 인코딩 문제
한국어 사이트 중 일부는 euc-kr 인코딩 사용
response.encoding 확인 후 필요하면 직접 지정:
  response.encoding = "euc-kr"
  또는
  text = response.content.decode("euc-kr")


## 5-2. 크롤링 주의사항

| 항목            | 권장 사항                                      |
| :-------------- | :--------------------------------------------- |
| **요청 간격**   | 요청 사이 `time.sleep(0.5~1)` 추가             |
| **robots.txt**  | `사이트주소/robots.txt` 확인 후 허용 여부 판단 |
| **User-Agent**  | 필요시 헤더에 User-Agent 설정                  |
| **이용약관**    | 크롤링 금지 조항 여부 확인                     |
| **과도한 요청** | 짧은 시간 내 대량 요청 자제                    |
| **데이터 활용** | 수집 데이터를 상업적으로 이용 시 저작권 확인   |

```python
# User-Agent 설정 예시
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}
response = requests.get(url, headers=headers, timeout=10)
```


In [28]:
def check_robots_txt(base_url):
    """사이트의 robots.txt 내용을 확인합니다."""
    robots_url = urljoin(base_url, '/robots.txt')
    try:
        resp = requests.get(robots_url, timeout=5)
        if resp.status_code == 200:
            print(f'robots.txt ({robots_url}):')
            print(resp.text[:500])
        else:
            print(f'robots.txt 없음 (상태 코드: {resp.status_code})')
    except Exception as e:
        print(f'확인 실패: {e}')

check_robots_txt(SITE_URL)


robots.txt 없음 (상태 코드: 404)


---
# 📋 BS4 간단 정리

## 요청 & 파싱
```python
response = requests.get(url, timeout=5)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
```

## 요소 탐색
| 메서드 | 설명 |
|:---|:---|
| `soup.find('태그')` | 첫 번째 요소 |
| `soup.find_all('태그')` | 모든 요소 리스트 |
| `soup.find('태그', class_='cls')` | 클래스 조건 (class_ 주의) |
| `soup.select_one('선택자')` | CSS 선택자로 첫 번째 |
| `soup.select('선택자')` | CSS 선택자로 모든 요소 |

## 텍스트 & 속성
```python
el.get_text(strip=True)          # 텍스트 추출
el.get_text(separator=' ')       # 구분자로 합치기
' '.join(el.stripped_strings)    # 공백 정제 후 합치기
el.get('href')                   # 속성값 (없으면 None)
el.get('href', '#')              # 속성값 (없으면 기본값)
el.attrs                         # 모든 속성 딕셔너리
```

## CSS 선택자 핵심 (html.parser 지원 범위)
```
✅ #id   .class   태그   부모 자손   부모 > 자식
✅ [attr=val]   [attr^="val"]   [attr$="val"]   [attr*="val"]
   ※ 속성값에 특수문자(.  /  : 등)가 있으면 반드시 따옴표로 감쌀 것
✅ :first-child
❌ :last-child, :nth-child(n)  →  select() + 인덱싱으로 대체
```

## :last-child, :nth-child 대체 패턴
```python
rows = soup.select('tbody tr')
rows[0]    # 첫 번째
rows[-1]   # 마지막
rows[n-1]  # n번째
```

## 안전 패턴
```python
el = soup.select_one('.title')
text = el.get_text(strip=True) if el else None
href = el.get('href') if el else None
full_url = urljoin(base_url, href) if href else None
```

---

> 🎉 수고하셨습니다! 실습 사이트: 👉 [crawl-da.netlify.app](https://crawl-da.netlify.app)
